<div style="background:linear-gradient(135deg,#0d1117,#13243b,#0f3a5f);padding:44px 38px;border-radius:16px;color:#f0f6fc;font-family:'Segoe UI',sans-serif;border:1px solid #30363d;">
  <div style="font-size:.8em;letter-spacing:3px;opacity:.65;text-transform:uppercase;margin-bottom:10px;">Attention Under Domain Shift · Explainability · Inference Only</div>
  <h1 style="font-size:2.0em;margin:0 0 12px 0;font-weight:700;line-height:1.25;color:#f0f6fc !important;">Dış Kümelerde Akciğer-Odak Oranı</h1>
  <h2 style="font-size:1.05em;font-weight:300;opacity:.82;margin:0 0 22px 0;line-height:1.5;color:#f0f6fc !important;">Dikkat örüntüsü dağıtım kayması altında korunuyor mu — yoksa yalnızca eğitim dağıtımına mı özgü?</h2>
  <hr style="border:0;border-top:1px solid #30363d;margin:0 0 18px 0;">
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;font-size:.88em;opacity:.88;">
    <div><b>Eğitim:</b> yok — kayıtlı kontrol noktaları</div>
    <div><b>Kümeler:</b> RSNA + NIH (yetişkin)</div>
    <div><b>Süre:</b> ~12 dakika</div>
    <div><b>Ek:</b> toplu dikkat haritaları, maske alanı denetimi</div>
  </div>
  <div style="margin-top:20px;padding:13px 16px;background:rgba(56,139,253,.10);border-left:4px solid #388bfd;border-radius:4px;font-size:.86em;line-height:1.55;">
    <b>Neden gerekli:</b> Akciğer-odak oranı şu ana dek yalnızca <i>iç</i> test kümesinde ölçüldü (A: 0,005 · B: 0,111 · C: 0,909). Bu çalışmanın ana iddiası bu farka dayandığından, örüntünün eğitim dağıtımına özgü bir olgu olmadığını göstermek zorunludur. Ayrıca C kolunun yüksek oranı kısmen tasarım gereğidir — ancak <b>yetişkin</b> görüntülerde segmentasyon zorlanırsa bu garanti zayıflar. Bu notebook her ikisini de sınar.
  </div>
</div>

## Kurulum

| # | Sekme | Kimlik |
|---|---|---|
| 1 | Notebooks | ablasyon notebook çıktısı (`ablation_vit_*.pth` + `ablation_predictions.npz`) |
| 2 | Competitions | `rsna-pneumonia-detection-challenge` |
| 3 | Datasets | `nih-chest-xrays/data` |

**Add Input → Notebooks → `segmentation-ablation-lung-focused-vit`** ile tek hamlede hem
kontrol noktaları hem de iç LFR değerleri gelir (karşılaştırma için kullanılır).
Eğitim veri kümesine gerek yoktur. GPU ve Internet açık olmalıdır.

### Ölçülenler

1. **Kol × küme LFR dağılımları** — iç test değerleriyle yan yana
2. **Eşleştirilmiş Wilcoxon testi** — aynı görüntü üç kolun hattından geçtiği için
   örneklemler eşleştirilmiştir; sıralı işaretli test doğru araçtır
3. **Toplu dikkat haritaları** — kol × küme ızgarası, akciğer konturu bindirilmiş
4. **LFR ve doğruluk ilişkisi** — akciğere bakan tahminler daha mı isabetli?
5. **Maske alanı denetimi** — ianpan yetişkin görüntülerde makul maskeler üretiyor mu?

In [ ]:
import os, sys, gc, io, json, glob, time, math, random, zipfile, warnings, types
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from collections import OrderedDict, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from sklearn.metrics import roc_curve, auc
from scipy import stats as sp_stats

# ----------------------------- AYARLAR -----------------------------
SEED               = 42     # ornek secimi - ablasyonla AYNI
MAX_PER_CLASS_LFR  = 250    # dis setlerden sinif basina goruntu
TAU                = 0.20   # LFR gurultu esigi - ablasyonla AYNI
ARMS               = ["raw", "roi", "lung"]
# --------------------------------------------------------------------

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ARM_LABEL = OrderedDict([("raw", "A · Segmentasyonsuz"),
                         ("roi", "B · Yalnizca RoI kirpma"),
                         ("lung", "C · Maske + RoI (onerilen)")])
ARM_COLOR = {"raw": "#B04A1E", "roi": "#C2900A", "lung": "#0D8FA2"}
MEDICAL_CMAP = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "medical", ["#000033", "#0033CC", "#00CCFF", "#FFFF00", "#FF4500"], N=256)

plt.rcParams.update({
    "figure.dpi": 130, "figure.facecolor": "white", "savefig.facecolor": "white",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8A9499", "axes.labelcolor": "#1B2327", "text.color": "#1B2327",
    "xtick.color": "#5A686F", "ytick.color": "#5A686F",
    "grid.color": "#D8DFE1", "grid.linewidth": 0.7, "legend.frameon": False,
})
WORK = "/kaggle/working"
print(f"Cihaz: {device} | sinif basina {MAX_PER_CLASS_LFR} goruntu | tau = {TAU}")

In [ ]:
# ── Girdiler ─────────────────────────────────────────────────────────────
INPUT = "/kaggle/input"

def find_dir_with(fname, root=INPUT):
    for r, _, files in os.walk(root):
        if fname in files:
            return r
    return None

ckpts = {}
for p in glob.glob(os.path.join(INPUT, "**", "*.pth"), recursive=True):
    b = os.path.basename(p).lower()
    for arm in ARMS:
        if f"_{arm}." in b:
            ckpts[arm] = p

RSNA_BASE = find_dir_with("stage_2_detailed_class_info.csv")
NIH_BASE  = find_dir_with("Data_Entry_2017.csv")

# Ic LFR degerleri (onceki kosum) - karsilastirma icin
LFR_INTERNAL = {}
for pat in ("ablation_predictions.npz", "ablation_predictions.zip"):
    hits = glob.glob(os.path.join(INPUT, "**", pat), recursive=True)
    if hits:
        try:
            z = zipfile.ZipFile(hits[0])
            for arm in ARMS:
                n = f"lfr__{arm}.npy"
                if n in z.namelist():
                    LFR_INTERNAL[arm] = np.load(io.BytesIO(z.read(n)))
        except Exception as e:
            print("Ic LFR okunamadi:", e)
        break
if not LFR_INTERNAL:
    loose = glob.glob(os.path.join(INPUT, "**", "lfr__*.npy"), recursive=True)
    for f in loose:
        LFR_INTERNAL[os.path.basename(f).split("__")[1][:-4]] = np.load(f)

print("Bulunanlar")
for arm in ARMS:
    print(f"  ckpt {arm:<5}: {ckpts.get(arm)}")
print(f"  RSNA : {RSNA_BASE}")
print(f"  NIH  : {NIH_BASE}")
print(f"  ic LFR: { {k: len(v) for k, v in LFR_INTERNAL.items()} or 'bulunamadi'}")

missing = [a for a in ARMS if a not in ckpts]
assert not missing, f"Kontrol noktasi eksik: {missing}"
assert RSNA_BASE or NIH_BASE, "Hicbir dis set bulunamadi."

In [ ]:
# ── Modeller ve segmentasyon ─────────────────────────────────────────────
def build_vit_inference(num_classes, dropout):
    m = models.vit_b_16(weights=None)
    in_f = m.heads.head.in_features
    m.heads.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes))
    return m

MODELS, CFG, CLASS_TO_IDX = {}, None, None
for arm in ARMS:
    ck = torch.load(ckpts[arm], map_location=device, weights_only=False)
    CFG, CLASS_TO_IDX = ck["config"], ck["class_to_idx"]
    m = build_vit_inference(CFG["num_classes"], CFG.get("dropout", 0.1)).to(device)
    m.load_state_dict(ck["model_state_dict"], strict=True)
    MODELS[arm] = m.eval()
    print(f"  {arm:<5} yuklendi (Val F1 {ck.get('best_val_f1'):.4f})")
PNEU_IDX = CLASS_TO_IDX["PNEUMONIA"]

eval_tf = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(CFG["mean"], CFG["std"]),
])

import transformers
from transformers import AutoModel
print("\nianpan/chest-x-ray-basic yukleniyor...")

def _load_seg():
    return AutoModel.from_pretrained("ianpan/chest-x-ray-basic",
                                     trust_remote_code=True).to(device).eval()

_of = getattr(transformers.modeling_utils.PreTrainedModel, "_finalize_model_loading", None)
try:
    if _of is not None:
        def _sf(model, *a, **k):
            if not hasattr(model, "all_tied_weights_keys"):
                model.all_tied_weights_keys = {}
            return _of(model, *a, **k)
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _sf
    seg_model = _load_seg()
except Exception as e:
    if "all_tied_weights_keys" in str(e):
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = {}
        seg_model = _load_seg()
    else:
        raise
finally:
    if _of is not None:
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _of
print("Segmentasyon modeli hazir.")

In [ ]:
# ── On-isleme: ablasyon notebook'u ile birebir ayni ──────────────────────
def load_image_any(path, short_max):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".dcm":
        import pydicom
        dcm = pydicom.dcmread(path)
        arr = dcm.pixel_array.astype(np.float32)
        arr -= arr.min()
        if arr.max() > 0:
            arr /= arr.max()
        if getattr(dcm, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
            arr = 1.0 - arr
        pil = Image.fromarray((arr * 255).astype(np.uint8)).convert("RGB")
    else:
        pil = Image.open(path).convert("RGB")
    W0, H0 = pil.size
    short = min(W0, H0)
    if short > short_max:
        s = short_max / short
        pil = pil.resize((int(round(W0 * s)), int(round(H0 * s))), Image.BILINEAR)
    return np.asarray(pil).astype(np.uint8), np.asarray(pil.convert("L"))


@torch.inference_mode()
def lung_mask_ianpan(gray_u8, out_hw):
    x = seg_model.preprocess(gray_u8)
    x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).float().to(device)
    logits = seg_model(x)["mask"]
    logits = F.interpolate(logits, size=out_hw, mode="bilinear", align_corners=False)
    pred = logits.argmax(dim=1)[0].cpu().numpy()
    return ((pred == 1) | (pred == 2)).astype(np.uint8)


def prep_arms(rgb_u8, lung_u8, cfg):
    H, W = lung_u8.shape
    short = min(H, W); S = cfg["img_size"]
    out = {}
    out["raw"] = (cv2.resize(rgb_u8, (S, S), interpolation=cv2.INTER_AREA),
                  cv2.resize(lung_u8, (S, S), interpolation=cv2.INTER_NEAREST).astype(np.uint8),
                  True)
    if lung_u8.sum() < 1:
        fb = cv2.resize(rgb_u8, (S, S), interpolation=cv2.INTER_AREA)
        ones = np.ones((S, S), np.uint8)
        out["roi"] = (fb, ones, False); out["lung"] = (fb, ones, False)
        return out
    dil = max(1, int(round(short * cfg["mask_dilate_frac"])))
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dil + 1, 2 * dil + 1))
    mask_d = cv2.dilate(lung_u8, kern, iterations=1)
    soft = mask_d.astype(np.float32)
    f = cfg["mask_feather"]
    if f and f >= 3:
        if f % 2 == 0:
            f += 1
        soft = cv2.GaussianBlur(soft, (f, f), 0)
    soft = np.clip(soft, 0.0, 1.0)[..., None]
    fill = (np.array([m * 255.0 for m in cfg["mean"]], dtype=np.float32)
            if cfg["fill_mode"] == "mean" else np.zeros(3, dtype=np.float32))
    masked = (rgb_u8.astype(np.float32) * soft +
              fill[None, None, :] * (1.0 - soft)).clip(0, 255).astype(np.uint8)
    ys, xs = np.where(mask_d > 0)
    y0, y1 = int(ys.min()), int(ys.max()); x0, x1 = int(xs.min()), int(xs.max())
    pad = int(round(short * cfg["roi_pad_frac"]))
    y0 = max(0, y0 - pad); x0 = max(0, x0 - pad)
    y1 = min(H - 1, y1 + pad); x1 = min(W - 1, x1 + pad)
    bh, bw = (y1 - y0 + 1), (x1 - x0 + 1)
    side = min(max(bh, bw), min(H, W))
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    ty0 = max(0, cy - side // 2); tx0 = max(0, cx - side // 2)
    ty1 = min(H, ty0 + side);     tx1 = min(W, tx0 + side)
    ty0 = max(0, ty1 - side);     tx0 = max(0, tx1 - side)
    m224 = cv2.resize(mask_d[ty0:ty1, tx0:tx1], (S, S),
                      interpolation=cv2.INTER_NEAREST).astype(np.uint8)
    out["roi"]  = (cv2.resize(rgb_u8[ty0:ty1, tx0:tx1], (S, S), interpolation=cv2.INTER_AREA),
                   m224, True)
    out["lung"] = (cv2.resize(masked[ty0:ty1, tx0:tx1], (S, S), interpolation=cv2.INTER_AREA),
                   m224, True)
    return out


class AttentionRollout:
    '''CLS -> yama dikkat akisi. compute() hem haritayi hem olasiligi doner
    (tek ileri gecis; ayrica tahmin icin ikinci gecis gerekmez).'''
    def __init__(self, model):
        self.model = model; self.attn_maps = []; self._hooks = []
        for layer in self.model.encoder.layers:
            sa = layer.self_attention
            orig = sa.forward
            def make_patched(orig_fwd):
                def patched(self_mod, *a, **kw):
                    kw["need_weights"] = True
                    kw["average_attn_weights"] = False
                    return orig_fwd(*a, **kw)
                return patched
            sa.forward = types.MethodType(make_patched(orig), sa)
            def make_hook(obj):
                def hook(m, inp, out):
                    if isinstance(out, tuple) and len(out) > 1 and out[1] is not None:
                        obj.attn_maps.append(out[1].detach().cpu())
                return hook
            self._hooks.append(sa.register_forward_hook(make_hook(self)))

    def remove(self):
        for h in self._hooks:
            h.remove()

    def compute(self, img_tensor):
        self.attn_maps = []
        self.model.eval()
        with torch.no_grad():
            out = self.model(img_tensor.unsqueeze(0).to(device))
        prob = torch.softmax(out, 1)[0, PNEU_IDX].item()
        N = self.attn_maps[0].size(-1)
        result = torch.eye(N)
        for attn in self.attn_maps:
            a = attn.squeeze(0)
            if a.dim() == 3:
                a = a.mean(0)
            a = a + torch.eye(N)
            a = a / (a.sum(-1, keepdim=True) + 1e-8)
            result = torch.matmul(a, result)
        m = result[0, 1:].numpy()
        n = int(round(m.size ** 0.5))
        m = m.reshape(n, n)
        m = (m - m.min()) / (m.max() - m.min() + 1e-8)
        m = cv2.resize(m.astype(np.float32), (CFG["img_size"], CFG["img_size"]))
        return (m - m.min()) / (m.max() - m.min() + 1e-8), prob


def lung_focus_ratio(sal, mask, tau=TAU):
    s = np.asarray(sal, np.float32).ravel(); mk = np.asarray(mask, np.float32).ravel()
    hi = s > tau
    if not hi.any() or s[hi].sum() < 1e-8:
        return float("nan")
    return float((s[hi] * mk[hi]).sum() / s[hi].sum())

ROLLOUT = {a: AttentionRollout(MODELS[a]) for a in ARMS}
print("Rollout hazir:", {a: len(ROLLOUT[a]._hooks) for a in ARMS})

In [ ]:
# ── Ornekleme: ablasyonla ayni tohum, ilk N ust kume ─────────────────────
def build_rsna_items(base, k, seed=SEED):
    info = pd.read_csv(os.path.join(base, "stage_2_detailed_class_info.csv")).drop_duplicates("patientId")
    img_dir = os.path.join(base, "stage_2_train_images")
    pos = info[info["class"] == "Lung Opacity"]["patientId"].tolist()
    neg = info[info["class"] == "Normal"]["patientId"].tolist()
    rng = random.Random(seed); rng.shuffle(pos); rng.shuffle(neg)
    kk = min(k, len(pos), len(neg))
    return ([(os.path.join(img_dir, p + ".dcm"), PNEU_IDX) for p in pos[:kk]] +
            [(os.path.join(img_dir, p + ".dcm"), 1 - PNEU_IDX) for p in neg[:kk]])

def build_nih_items(base, k, seed=SEED):
    df = pd.read_csv(os.path.join(base, "Data_Entry_2017.csv"))
    lab = df["Finding Labels"].astype(str)
    is_p = lab.apply(lambda s: "Pneumonia" in s.split("|"))
    is_n = lab.apply(lambda s: s.strip() == "No Finding")
    index = {}
    for r, _, files in os.walk(base):
        for f in files:
            if f.lower().endswith(".png"):
                index[f] = os.path.join(r, f)
    pf = lambda m: [index[n] for n in df[m]["Image Index"].tolist() if n in index]
    pos, neg = pf(is_p), pf(is_n)
    rng = random.Random(seed); rng.shuffle(pos); rng.shuffle(neg)
    kk = min(k, len(pos), len(neg))
    return [(p, PNEU_IDX) for p in pos[:kk]] + [(p, 1 - PNEU_IDX) for p in neg[:kk]]

EXTERNAL = []
if RSNA_BASE: EXTERNAL.append(("RSNA", build_rsna_items(RSNA_BASE, MAX_PER_CLASS_LFR)))
if NIH_BASE:  EXTERNAL.append(("NIH",  build_nih_items(NIH_BASE, MAX_PER_CLASS_LFR)))
for n, it in EXTERNAL:
    print(f"{n}: {len(it)} goruntu")

In [ ]:
# ── Ana dongu: segmentasyon -> uc kol -> rollout -> LFR ──────────────────
S = CFG["img_size"]
REC = []                                    # goruntu bazinda kayit
AGG = {(n, a): {"cam": np.zeros((S, S)), "mask": np.zeros((S, S)), "n": 0}
       for n, _ in EXTERNAL for a in ARMS}

for name, items in EXTERNAL:
    print(f"\n>>> {name} ({len(items)} goruntu x 3 kol)...")
    t0 = time.time()
    for i, (path, label) in enumerate(items):
        try:
            rgb, gray = load_image_any(path, CFG["orig_short_max"])
            lung = lung_mask_ianpan(gray, gray.shape)
            arms = prep_arms(rgb, lung, CFG)
        except Exception:
            continue
        row = {"kume": name, "dosya": os.path.basename(path), "etiket": int(label),
               "maske_alani": float(lung.mean()), "fallback": (not arms["lung"][2])}
        for a in ARMS:
            img224, m224, _ = arms[a]
            x = eval_tf(Image.fromarray(img224))
            sal, prob = ROLLOUT[a].compute(x)
            v = lung_focus_ratio(sal, m224)
            row[f"lfr_{a}"] = v
            row[f"p_{a}"] = prob
            if not np.isnan(v):
                AGG[(name, a)]["cam"] += sal
                AGG[(name, a)]["mask"] += m224
                AGG[(name, a)]["n"] += 1
        REC.append(row)
        if (i + 1) % 100 == 0:
            print(f"    {i+1}/{len(items)} ({time.time()-t0:.0f} sn)")
    print(f"  bitti: {time.time()-t0:.0f} sn")

df = pd.DataFrame(REC)
print(f"\nToplam kayit: {len(df)} | geri cekilme: {int(df['fallback'].sum())}")

## Sonuçlar

Aynı görüntü üç kolun hattından geçtiği için örneklemler **eşleştirilmiştir**;
kollar arası karşılaştırmada bağımsız örneklem testi değil, **Wilcoxon işaretli sıra
testi** kullanılır. Etki büyüklüğü olarak medyan farkı ve eşleştirilmiş bootstrap
güven aralığı verilir.

In [ ]:
# ── LFR ozet tablosu: ic test + dis kumeler ──────────────────────────────
rows = []
for a in ARMS:
    if a in LFR_INTERNAL:
        v = LFR_INTERNAL[a]
        rows.append({"kume": "Ic test", "kol": a, "n": len(v), "ort": v.mean(),
                     "ss": v.std(), "medyan": np.median(v),
                     "<0,05": float((v < 0.05).mean()), ">0,50": float((v > 0.50).mean())})
for name, _ in EXTERNAL:
    sub = df[df.kume == name]
    for a in ARMS:
        v = sub[f"lfr_{a}"].dropna().values
        rows.append({"kume": name, "kol": a, "n": len(v), "ort": v.mean(),
                     "ss": v.std(), "medyan": np.median(v),
                     "<0,05": float((v < 0.05).mean()), ">0,50": float((v > 0.50).mean())})
df_lfr = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("=" * 92)
print("  AKCIGER-ODAK ORANI  (tau = 0,20)")
print("=" * 92)
print(df_lfr.to_string(index=False))
print("=" * 92)

# ── Eslestirilmis Wilcoxon ───────────────────────────────────────────────
PAIRS = [("lung", "raw"), ("lung", "roi"), ("roi", "raw")]
rows = []
rng = np.random.default_rng(SEED)
for name, _ in EXTERNAL:
    sub = df[df.kume == name]
    for a, b in PAIRS:
        va = sub[f"lfr_{a}"].values; vb = sub[f"lfr_{b}"].values
        ok = ~(np.isnan(va) | np.isnan(vb))
        va, vb = va[ok], vb[ok]
        d = va - vb
        try:
            stat, p = sp_stats.wilcoxon(va, vb)
        except Exception:
            stat, p = np.nan, np.nan
        boot = np.array([np.median(d[rng.integers(0, len(d), len(d))]) for _ in range(2000)])
        rows.append({"kume": name, "karsilastirma": f"{a} - {b}", "n": len(d),
                     "medyan fark": float(np.median(d)),
                     "GA alt": float(np.percentile(boot, 2.5)),
                     "GA ust": float(np.percentile(boot, 97.5)),
                     "Wilcoxon p": p,
                     "ustun oran": float((d > 0).mean())})
df_w = pd.DataFrame(rows)
print("\n" + "=" * 108)
print("  ESLESTIRILMIS KARSILASTIRMA (Wilcoxon isaretli sira)")
print("=" * 108)
print(df_w.to_string(index=False))
print("=" * 108)

In [ ]:
# ── Figur 1: LFR dagilimlari, kume x kol ─────────────────────────────────
panels = ["Ic test"] + [n for n, _ in EXTERNAL]
fig, axes = plt.subplots(1, len(panels), figsize=(4.3 * len(panels), 4.4), squeeze=False)
rngj = np.random.default_rng(0)
for ax, pname in zip(axes[0], panels):
    for k, a in enumerate(ARMS):
        if pname == "Ic test":
            v = LFR_INTERNAL.get(a, np.array([]))
        else:
            v = df[df.kume == pname][f"lfr_{a}"].dropna().values
        if len(v) == 0:
            continue
        ax.scatter(k + rngj.uniform(-0.13, 0.13, len(v)), v, s=9, alpha=.32,
                   color=ARM_COLOR[a], edgecolor="none", zorder=3)
        bp = ax.boxplot([v], positions=[k], widths=.42, showfliers=False,
                        patch_artist=True, zorder=4,
                        medianprops=dict(color="white", lw=2),
                        boxprops=dict(facecolor=ARM_COLOR[a], alpha=.85, edgecolor="none"),
                        whiskerprops=dict(color=ARM_COLOR[a], lw=1.2),
                        capprops=dict(color=ARM_COLOR[a], lw=1.2))
        ax.text(k, 1.045, f"{np.median(v):.3f}", ha="center", fontsize=8.6, zorder=5)
    ax.set_xticks(range(len(ARMS))); ax.set_xticklabels(["A", "B", "C"])
    ax.set_ylim(-0.04, 1.10); ax.set_title(pname); ax.set_ylabel("Akciger-odak orani")
    ax.yaxis.grid(True, alpha=.5); ax.set_axisbelow(True)
    ax.axhline(0.5, color="#8A9499", ls=":", lw=1, zorder=2)
from matplotlib.patches import Patch
fig.legend(handles=[Patch(facecolor=ARM_COLOR[a], label=ARM_LABEL[a]) for a in ARMS],
           loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.02), fontsize=9)
fig.suptitle("Akciger-odak orani: ic testte gorulen ayrisma dis kumelerde de korunuyor mu?",
             fontsize=12, fontweight="bold")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(os.path.join(WORK, "extlfr_fig_01_distributions.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figur 2: toplu dikkat haritalari (kume x kol) ────────────────────────
nrow = len(EXTERNAL)
fig, axes = plt.subplots(nrow, len(ARMS), figsize=(4.0 * len(ARMS), 4.1 * nrow), squeeze=False)
for r, (name, _) in enumerate(EXTERNAL):
    for c, a in enumerate(ARMS):
        d = AGG[(name, a)]
        ax = axes[r][c]
        if d["n"] == 0:
            ax.axis("off"); continue
        cam = d["cam"] / d["n"]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        ax.imshow(cam, cmap=MEDICAL_CMAP, vmin=0, vmax=1)
        ax.contour(d["mask"] / d["n"], levels=[0.4], colors="white", linewidths=1.8)
        v = df[df.kume == name][f"lfr_{a}"].dropna().values
        ax.set_title(f"{name} — {ARM_LABEL[a]}\nLFR = {v.mean():.3f} ± {v.std():.3f}  (n={d['n']})",
                     fontsize=9.5, color=ARM_COLOR[a], fontweight="bold")
        ax.axis("off")
fig.suptitle("Toplu dikkat haritalari — beyaz kontur: ortalama akciger maskesi",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "extlfr_fig_02_aggregate.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── LFR ile dogruluk iliskisi ────────────────────────────────────────────
rows = []
for name, _ in EXTERNAL:
    sub = df[df.kume == name]
    for a in ARMS:
        v = sub[f"lfr_{a}"].values; p = sub[f"p_{a}"].values; y = sub["etiket"].values
        ok = ~np.isnan(v)
        v, p, y = v[ok], p[ok], y[ok]
        correct = ((p >= 0.5).astype(int) == y)
        if correct.sum() < 5 or (~correct).sum() < 5:
            continue
        u, pw = sp_stats.mannwhitneyu(v[correct], v[~correct], alternative="two-sided")
        rows.append({"kume": name, "kol": a,
                     "LFR dogru": float(v[correct].mean()),
                     "LFR yanlis": float(v[~correct].mean()),
                     "fark": float(v[correct].mean() - v[~correct].mean()),
                     "n dogru": int(correct.sum()), "n yanlis": int((~correct).sum()),
                     "MWU p": float(pw)})
df_corr = pd.DataFrame(rows)
print("=" * 96)
print("  DIKKAT AKCIGERDEYSE TAHMIN DAHA MI ISABETLI?")
print("=" * 96)
print(df_corr.to_string(index=False) if len(df_corr) else "  (yeterli veri yok)")
print("=" * 96)

# ── Segmentasyon davranisi: maske alani denetimi ─────────────────────────
print("\n" + "=" * 84)
print("  MASKE ALANI DENETIMI  (tam karede akciger orani; makul araligin %15-45 oldugu varsayilir)")
print("=" * 84)
rows = []
for name, _ in EXTERNAL:
    v = df[df.kume == name]["maske_alani"].values
    rows.append({"kume": name, "n": len(v), "ort": v.mean(), "ss": v.std(),
                 "medyan": np.median(v), "min": v.min(), "maks": v.max(),
                 "supheli (<0,15 | >0,45)": float(((v < 0.15) | (v > 0.45)).mean()),
                 "cok kucuk (<0,08)": float((v < 0.08).mean())})
df_mask = pd.DataFrame(rows)
print(df_mask.to_string(index=False))
print("=" * 84)

In [ ]:
# ── Kayit ────────────────────────────────────────────────────────────────
df.to_csv(os.path.join(WORK, "extlfr_per_image.csv"), index=False)
df_lfr.to_csv(os.path.join(WORK, "extlfr_summary.csv"), index=False)
df_w.to_csv(os.path.join(WORK, "extlfr_wilcoxon.csv"), index=False)
if len(df_corr):
    df_corr.to_csv(os.path.join(WORK, "extlfr_vs_correctness.csv"), index=False)
df_mask.to_csv(os.path.join(WORK, "extlfr_mask_area.csv"), index=False)
np.savez_compressed(os.path.join(WORK, "extlfr_aggregate_maps.npz"),
                    **{f"{n}__{a}__cam": AGG[(n, a)]["cam"] / max(AGG[(n, a)]["n"], 1)
                       for n, _ in EXTERNAL for a in ARMS},
                    **{f"{n}__{a}__mask": AGG[(n, a)]["mask"] / max(AGG[(n, a)]["n"], 1)
                       for n, _ in EXTERNAL for a in ARMS})

print("\n" + "=" * 72)
print("  OZET")
print("=" * 72)
for name, _ in EXTERNAL:
    s = " | ".join(f"{a}={df[df.kume==name][f'lfr_{a}'].mean():.3f}" for a in ARMS)
    print(f"  {name:<6} LFR: {s}")
if LFR_INTERNAL:
    s = " | ".join(f"{a}={LFR_INTERNAL[a].mean():.3f}" for a in ARMS if a in LFR_INTERNAL)
    print(f"  {'Ic':<6} LFR: {s}")
print("=" * 72)
for f in sorted(os.listdir(WORK)):
    if f.startswith("extlfr"):
        print(f"  {f:<36} {os.path.getsize(os.path.join(WORK, f))/1e6:>7.2f} MB")

## Sonucun okunması

**Beklenen örüntü:** iç testteki sıralama (C ≫ B > A) dış kümelerde de korunmalıdır.
Korunuyorsa, dikkat farkı eğitim dağıtımına özgü bir olgu değil, ön-işlemenin doğrudan
sonucudur — makalenin ana iddiası dağıtım kayması altında da geçerlidir.

**C kolunda düşüş olursa:** yetişkin görüntülerde segmentasyon zorlanıyor ve maske
patoloji bölgelerini dışarıda bırakıyor olabilir. Maske alanı denetimindeki "şüpheli"
oranı bu durumda yükselmiş olmalıdır; iki sonuç birlikte yorumlanmalıdır.

**A kolunda yükseliş olursa:** yetişkin görüntülerde akciğer karenin daha büyük bir
kısmını kapladığından rastgele dikkat bile daha yüksek LFR üretir. Bu nedenle A kolunun
değeri, aynı kümedeki **maske alanı ortalamasıyla** karşılaştırılmalıdır: LFR ≈ maske
alanı ise dikkat esasen rastgeledir; LFR ≪ maske alanı ise dikkat akciğerden aktif
olarak kaçıyor demektir.

> **Not.** Bu ölçüm tek eğitim tohumuyla (tohum 42 kontrol noktaları) yapılmıştır.
> Çoklu tohum çalışması AUC farkının tohuma duyarlı olduğunu göstermişti; ancak buradaki
> etki büyüklüğü (≈ 0,005 → 0,909) iç testte ölçülen tohum gürültüsünün iki büyüklük
> mertebesi üzerindedir, dolayısıyla tohuma atfedilemez.